# Feature selection: lab notebook

**Machine Learning (5CS4-05), Statistical learning theory, features, model selection**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

### 1. A data set with known structure: F1-F3 matter, F4 copies F1, F5 copies F2, F6-F8 are noise

In [ ]:
# 1. A data set with known structure: F1-F3 matter, F4 copies F1, F5 copies F2, F6-F8 are noise
rng = np.random.default_rng(1)
n = 200
a, b, c = rng.normal(size=(3, n))
X = np.column_stack([a, b, c, a + 0.3 * rng.normal(size=n), b + 0.3 * rng.normal(size=n), rng.normal(size=(n, 3))])
y = 3 * a - 2 * b + 1.5 * c + rng.normal(size=n)
names = [f"F{i + 1}" for i in range(8)]

def cv(cols):
    return -cross_val_score(LinearRegression(), X[:, cols], y, cv=5, scoring="neg_mean_squared_error").mean()

### 2. Filter: rank by univariate F-score

In [ ]:
# 2. Filter: rank by univariate F-score
sk = SelectKBest(f_regression, k=3).fit(X, y)
sel = list(np.where(sk.get_support())[0])
print("filter keeps:", [names[i] for i in sel], " CV MSE:", round(cv(sel), 3))

### 3. Wrapper: forward selection with a linear model

In [ ]:
# 3. Wrapper: forward selection with a linear model
fw = SequentialFeatureSelector(LinearRegression(), n_features_to_select=3, direction="forward", cv=5).fit(X, y)
sel = list(np.where(fw.get_support())[0])
print("forward keeps:", [names[i] for i in sel], " CV MSE:", round(cv(sel), 3))

### 4. Embedded: Lasso on standardised features

In [ ]:
# 4. Embedded: Lasso on standardised features
Z = StandardScaler().fit_transform(X)
for alpha in (0.05, 0.3, 1.0):
    w = Lasso(alpha=alpha).fit(Z, y).coef_
    print(f"alpha = {alpha}: non-zero =", [names[i] for i in np.nonzero(w)[0]])

### 5. How many subsets would exhaustive search need?

In [ ]:
# 5. How many subsets would exhaustive search need?
print("subsets of 8 features:", 2 ** 8 - 1, " forward selection of 3 fits:", 8 + 7 + 6, "models")

## Expected output

Running all cells above should print:

```
filter keeps: ['F1', 'F3', 'F4']  CV MSE: 3.748
forward keeps: ['F1', 'F2', 'F3']  CV MSE: 1.01
alpha = 0.05: non-zero = ['F1', 'F2', 'F3', 'F4', 'F5', 'F6']
alpha = 0.3: non-zero = ['F1', 'F2', 'F3', 'F4']
alpha = 1.0: non-zero = ['F1', 'F2', 'F3']
subsets of 8 features: 255  forward selection of 3 fits: 21 models
```

## Practice exercises

Try these in the cells below.

1. Use SelectKBest with k = 3 and compare it with the features RFE chooses.
2. Increase the Lasso alpha step by step and record when each coefficient hits zero.
3. Add two more noise columns. Does the filter ranking still put the informative columns first?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
